In [ ]:
import random
import numpy as np
import torch

def set_seed(seed=32):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Make cuDNN deterministic
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # Request deterministic PyTorch algorithms
    # torch.use_deterministic_algorithms(True, warn_only=True)

    print(f"Random seed set to {seed}")

set_seed(32)

Random seed set to 32


In [ ]:
import torch
import torch.nn as nn

In [ ]:
'''
|Encoding :
    During encoding we find the important features of image

|Decoding :
    During Decoding we use that features to construct segmented image

|skip connection :
    We use skip connection to forward the encoder blocks inforamtion to Decoder
    blocks which help the decoders to remember the every detail captured by encoder.This helps
    decoder to construct detailed and accurate segmented image

    ## In U-net during skip connection we use concatination instead of adding (these are different from Resnet skip connections)

|steping in Encoder path :
    Every time we move one step down during encoding we use max_pool to reduce image resolution

|steping in Decoder path :
    When we move one step above in Decoder block we use Transpose convolution, We construct
    segmented image in decoding process using the high level details of bottleneck volume and
    low level details from the encoder blocks using the skip connections

|Why volume height and width decreases but the depth increase as we move down ?
    * small volume in shallow layers because shallow layeres contain low level inforamtion about image like edges,corners,straight lines,faces,doors location
    * But as we move down deeper layers use low level information to detect high level features in image like what image is telling
    (why smiley face,why balck door) or say intelligent details
    * high depth volumes represent that we contain detailed information | High Depth => high details (what and why) |
    * Low depth volumes represent that we contain low level information | Low depth => low details (where)|

    ## thats why the bottleneck layer contain most important details of image (intelligent information)

'''

class U_NET(nn.Module):

    def block(self, in_c, out_c, dropout_p=0.0):
        layers = [
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        ]
        if dropout_p > 0 :
            # the drop out layer sets between two layers and act as filter which pass
            # the activation of onyl activated neurons from it and for deactivated neurons
            # it passes zero as their activation

            layers.append(nn.Dropout2d(p=dropout_p)) # applying dropout to only blocks where dropout_p is given or say greater than 0

        return nn.Sequential(*layers) # * is unpaking all layers from layers list and putting them in nn.sequential container

    def __init__(self):
        super().__init__()

        self.pool = nn.MaxPool2d(2,2)

        # Enocder path
        self.e1 = self.block(3,64)
        self.e2 = self.block(64,128)
        self.e3 = self.block(128,256)
        self.e4 = self.block(256,512,0.3)

        # bottle neck
        self.bottle_neck = self.block(512,1024,0.5)
        # probability above like, the neurons whose random generated value is less than 0.5 will be turned off

        # Decoder Path
        # In decoder path when we move one step up we will use transpose convolution
        self.d1u = nn.ConvTranspose2d(1024,512,2,stride=2)
        self.d1 = self.block(1024,512) # 1024 => (512 skip connection) and (512 original)

        self.d2u = nn.ConvTranspose2d(512,256,2,stride=2)
        self.d2 = self.block(512,256)

        self.d3u = nn.ConvTranspose2d(256,128,2,stride=2)
        self.d3 = self.block(256,128)

        self.d4u = nn.ConvTranspose2d(128,64,2,stride=2)
        self.d4 = self.block(128,64)

        self.final = nn.Conv2d(64,2,1)

    def forward (self,x):
        # encoding
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        # bottle neck
        bottle_neck = self.bottle_neck(self.pool(e4))

        # Decoding
        d1u = self.d1u(bottle_neck)
        # first we join skip connection tensor and transpose conv tensor and then put into d1
        d1 = self.d1(torch.cat([e4,d1u],dim=1)) # skip connection | [batch,channel,height,width] dim=1 means concatenate along channels i.e (256+256) resulting channels become 512

        d2u = self.d2u(d1)
        d2 = self.d2(torch.cat([e3,d2u],dim=1))

        d3u = self.d3u(d2)
        d3 = self.d3(torch.cat([e2,d3u],dim=1))

        d4u = self.d4u(d3)
        d4 = self.d4(torch.cat([e1,d4u],dim=1))
        # puting output into 1x1 convolution
        out = self.final(d4)

        return out

model = U_NET()
model

U_NET(
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (e1): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
  )
  (e2): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
  )
  (e3): Sequential(
    (0): Conv2d(128,

In [ ]:
from torchvision.datasets import OxfordIIITPet
from torchvision.transforms import v2

# Download and load the dataset
train_set = OxfordIIITPet(
    root="./data",
    split="trainval",
    target_types="segmentation",
    download=True
)

test_set = OxfordIIITPet(
    root="./data",
    split="test",
    target_types="segmentation",
    download=True
)

100%|██████████| 792M/792M [00:16<00:00, 47.4MB/s] 
100%|██████████| 19.2M/19.2M [00:00<00:00, 19.8MB/s]


In [ ]:
'''
(1): The data set downloaded has bigger resolution and we will reduce the resolution to 128,128.
(2): resolution reduction happen like : we will divide the image into multiple parts say 128x128 each part contain multiple pixels we will perform
     operation on all pixel under particular part (average or addition) which return a single pixel value, this resulting single pixel will be used
     instead of all pixel under particular part.
(3): The interpolationMode decide which operation to perform we can set interpolation using below code, InterpolationMode.NEAREST says that take the
     Nearest pixel value (for sake of simplicity in this dataset you can say that either label the pixel as background or mask)
(4): I used v2.ToImage() because the PIL and numpy store image array as (Height, Width, Channels) but torch store images as (Channels, Height, Width)
     To proccess the Image on GPU, also pytorch model also want images as (Channels, Height, Width). So every time you want to send PIL or numpy image
     array to model convert it into pytorch image tensor
'''
from PIL.Image import Transform
from torchvision import transforms
from torchvision.transforms import InterpolationMode

image_transform = v2.Compose([
    v2.Resize((128,128)),
    v2.ToImage(),
    v2.ToDtype(torch.float32,scale=True) # scale = True => scale the pixel values between 0 and 1
])

mask_transform = v2.Compose([
    # we write this line first because v2.Resize() need 3D image and mask image of data set is 2D[height,width] so we convert it into [1,height,width]
    # image ramain grayscale we just added another dimension the channel dimension not included in data set images
    # we doing this just for Resize() function to work we again remove the channel dimension to make mask image 2D in below cell using squeeze(0) the reasin is also written there
    v2.ToImage(),
    v2.Resize((128,128),interpolation=InterpolationMode.NEAREST)
])


In [ ]:
# These tranformations are for data augementation to prevent model from overfitting
train_transforms = v2.Compose([
    # Joint spatial transforms (applied identically to image AND mask)
    # v2.RandomResizedCrop(size=(256, 256), scale=(0.75, 1.0), antialias=True),
    # v2.RandomHorizontalFlip(p=0.5),
    # v2.RandomRotation(degrees=20),

    # Photometric transforms (automatically skipped for tv_tensors.Mask)
    # v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    # v2.GaussianBlur(kernel_size=(3, 3), sigma=(0.1, 1.5)),

    # Convert image to normalized float (mask stays untouched)
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


# applying same normalization to validation set i will also have to use same on test set
# or any new image
val_transforms = v2.Compose([
    v2.Resize(size=(256, 256), antialias=True),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


In [ ]:
from torch.utils.data import Dataset
from torchvision import tv_tensors

class DATA_WRAPPER(Dataset):
  def __init__(self,dataset,transform=None):
      self.dataset = dataset
      self.transform = transform

  def __len__(self):
        return len(self.dataset)

  def __getitem__(self, index):
    image,mask = self.dataset[index]

    image = image_transform(image)
    mask = mask_transform(mask)

    # we are making mask and image a tv_tensors mask and image so that our transform donot
    # perform photometric tranformation on mask,because photometric transformation destory a discrete
    # mask containing [0,1] pixels by changing contrast and more,mask remain non-RGB but not-more remain descrete
    # i.e:(from just black and white mask to gray,black and more shades)
    # How this tv_tensors.Mask prevent transform from phototmetric Transformation ?
    # The v2 transform of pytorch is smart when it sees a tv_tensors mask it automatically prevent
    # any photometric tranformation or other transformation that changes the discrete mask . v2
    # transform has this built-in ability
    image = tv_tensors.Image(image)
    mask = tv_tensors.Mask(mask)


    # the below augementation transformations will apply like every time when an image
    # mask is picked from data set by Data loader the same augementaion transformation will
    # will be applied to both mask,image

    if self.transform is not None :
        image,mask = self.transform(image,mask)
    # removing first dimension because there is no need for RGb mask we want binary mask color (2D image) e.g black or white (black for background and white for
    # pet), also pytoech loss function need mask a 2D image.

    mask = mask.squeeze(0).long() # .long() changes float data type to integer so that each pixel has dicrete value 0 or 1 not 0.5,0.9 etc

    # (mask == 1) is a loop structure (the pet dataset contain three mask classes 1 for object, 2 for background, 3 for outline) so mask == 1 returns
    # True if mask label is 1 and False if mask label is 2 or 3. .long() convert the True and false values into 1 and 0 or say convert boolean data types
    # into integer. After this our mask variable contain object pixels as 1 and background pixels as 0 in easy words mask is now applied on object
    # and it doesnot contain the outline.| we did all this stuff for removing outline in mask
    mask = (mask == 1).long()

    return image,mask





In [ ]:
from torch.utils.data import random_split

train_set = DATA_WRAPPER(train_set,train_transforms)
test_set = DATA_WRAPPER(test_set,val_transforms)

vald_size = int(0.2*(len(test_set)))
test_size = len(test_set)-vald_size

test_subset,vald_subset = random_split(
    test_set,
    [test_size,vald_size],
    generator=torch.Generator().manual_seed(32)
)


In [ ]:
from torch.utils.data import DataLoader

g = torch.Generator()
g.manual_seed(32)

train_loader = DataLoader(
    train_set,
    batch_size=16,
    shuffle=True,
    num_workers=0, # num of cores want to assign for data loading
    generator = g
)

vald_loader = DataLoader(
    vald_subset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    generator = g
)

In [ ]:
images, masks = next(iter(train_loader))

print("Images:", images.shape)
print("Masks:", masks.shape)

print("Image dtype:", images.dtype)
print("Mask dtype:", masks.dtype)

print("Image range:", images.min().item(), images.max().item())
print("Mask labels:", torch.unique(masks))

Images: torch.Size([16, 3, 128, 128])
Masks: torch.Size([16, 128, 128])
Image dtype: torch.float32
Mask dtype: torch.int64
Image range: -2.1179039478302 2.640000104904175
Mask labels: tensor([0, 1])


In [ ]:
model = U_NET()
model = nn.DataParallel(model)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
model.to(device)

cuda


DataParallel(
  (module): U_NET(
    (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (e1): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
    (e2): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(

In [ ]:
epochs = 40

best_val_loss = float(0.2)
save_path = '/kaggle/working/best_model.pth'
total_saved_models = 0

for epoch in range(epochs):
    # --- TRAINING ---
    model.train()
    total_train_loss = 0.0

    for batch_idx, (images, masks) in enumerate(train_loader):
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()  # Fixed: Reset gradients for EVERY batch
        out = model(images)
        loss = criterion(out, masks)
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    # --- VALIDATION ---
    model.eval()
    total_val_loss = 0.0  # Fixed: Separate variable for validation loss

    with torch.no_grad():
        for images, masks in vald_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            loss = criterion(outputs, masks)
            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(vald_loader)

    if avg_val_loss <= best_val_loss :
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), save_path)
        print(f"Epoch {epoch+1}: New best model saved! (Val Loss: {best_val_loss:.4f})")
        # now updating the best_val_loss so that if model got more accurate then new limit it should again be saved in new save_path
        best_val_loss = float(0.15)
        save_path = '/kaggle/working/best_model2.pth'

    print(f"Epoch [{epoch+1}/{epochs}] -> train_loss: {avg_train_loss:.4f} | val_loss: {avg_val_loss:.4f}", flush=True)


Epoch [1/40] -> train_loss: 0.4361 | val_loss: 0.5964
Epoch [2/40] -> train_loss: 0.3644 | val_loss: 0.7053
Epoch [3/40] -> train_loss: 0.3220 | val_loss: 0.4631
Epoch [4/40] -> train_loss: 0.2903 | val_loss: 0.8579
Epoch [5/40] -> train_loss: 0.2642 | val_loss: 0.5353
Epoch [6/40] -> train_loss: 0.2418 | val_loss: 0.5747
Epoch [7/40] -> train_loss: 0.2273 | val_loss: 0.9460
Epoch [8/40] -> train_loss: 0.2181 | val_loss: 1.2497
Epoch [9/40] -> train_loss: 0.2035 | val_loss: 0.9580
Epoch [10/40] -> train_loss: 0.1961 | val_loss: 0.4824
Epoch [11/40] -> train_loss: 0.1851 | val_loss: 0.6303
Epoch [12/40] -> train_loss: 0.1827 | val_loss: 1.0315
Epoch [13/40] -> train_loss: 0.1699 | val_loss: 0.6195
Epoch [14/40] -> train_loss: 0.1597 | val_loss: 0.5869
Epoch [15/40] -> train_loss: 0.1527 | val_loss: 0.8000
Epoch [16/40] -> train_loss: 0.1478 | val_loss: 0.6428
Epoch [17/40] -> train_loss: 0.1426 | val_loss: 0.9060


In [ ]:
import os
from IPython.display import FileLink

# Change directory to /kaggle/working so FileLink resolves the relative path properly
os.chdir('/kaggle/working')

# Displays a clickable link directly below the cell
FileLink('best_model.pth')



In [ ]:
FileLink('best_model2.pth')